# Build & Save Parquet Files
Runs once. Produces everything the modelling notebook needs.

**Outputs**
- `data/global_counts.parquet` — global lemma counts across all titles
- `data/subject_counts/XX.pkl` — per-subject (year_word, year_total) cache
- `data/word_year_subject.parquet` — flat modelling table (word × subject × year)
- `data/subject_meta.parquet` — subject code → name + title count lookup

In [1]:
import re
import os
import pickle
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from tqdm.auto import tqdm

# ── paths ──────────────────────────────────────────────────────────────────
DATA_PATH   = Path.cwd().parent.parent / "data" / "processed" / "dataset_filled_subjects_new.csv"
OUT_DIR     = Path("data")
SUBJ_DIR    = OUT_DIR / "subject_counts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBJ_DIR.mkdir(parents=True, exist_ok=True)

print("Paths ready")

Paths ready


C:\Users\Work\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── load English-filtered data (assumes keyword_trends_english.pkl exists) ──
# If not, re-run the language-filter cell from the exploration notebook first.
df_en = pd.read_pickle("data/keyword_trends_english.pkl")
df_en["year"] = df_en["year"].astype(int)
df_en["code"] = df_en["subject_code"].astype(int).astype(str).str.zfill(2)

print(f"Loaded {len(df_en):,} English titles")
print(f"Subjects: {df_en['code'].nunique()}")

Loaded 208,150 English titles
Subjects: 63


In [3]:
# ── spacy setup ────────────────────────────────────────────────────────────
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

KEEP_POS = {"NOUN", "ADJ", "VERB", "PROPN"}

def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def tokenize_doc(doc):
    return [
        token.lemma_ for token in doc
        if token.pos_ in KEEP_POS
        and not token.is_stop
        and len(token.lemma_) >= 4
    ]

print("spacy loaded")

spacy loaded


In [4]:
# ── 1. GLOBAL COUNTS ───────────────────────────────────────────────────────
# One pass over all 208k titles. ~5-10 min.
# Skip if already saved.

GLOBAL_PATH = OUT_DIR / "global_counts.parquet"

if GLOBAL_PATH.exists():
    print("global_counts.parquet already exists, loading...")
    _g         = pd.read_parquet(GLOBAL_PATH)
    global_counts = Counter(dict(zip(_g["lemma"], _g["count"])))
    global_total  = int(_g["count"].sum())
else:
    print("Building global counts...")
    global_counts = Counter()
    global_total  = 0

    all_titles = [strip_accents(str(t).lower()) for t in df_en["thesis"]]

    for doc in tqdm(nlp.pipe(all_titles, batch_size=512, n_process=1),
                    total=len(all_titles), desc="global tokenize"):
        toks = tokenize_doc(doc)
        global_counts.update(toks)
        global_total += len(toks)

    # save as parquet (lemma, count)
    pd.DataFrame(
        global_counts.most_common(),
        columns=["lemma", "count"]
    ).to_parquet(GLOBAL_PATH, index=False)

    print(f"Saved {GLOBAL_PATH}")

print(f"Global vocabulary: {len(global_counts):,} lemmas | {global_total:,} total tokens")

global_counts.parquet already exists, loading...
Global vocabulary: 41,384 lemmas | 1,209,297 total tokens


In [5]:
# ── 2. PER-SUBJECT COUNTS ──────────────────────────────────────────────────
# Saves one .pkl per subject under data/subject_counts/.
# Already-processed subjects are skipped automatically.

all_subjects = sorted(df_en["code"].unique())
print(f"Processing {len(all_subjects)} subjects...")

for subj in tqdm(all_subjects, desc="subjects"):
    out_path = SUBJ_DIR / f"{subj}.pkl"
    if out_path.exists():
        continue

    sub_df       = df_en[df_en["code"] == subj].reset_index(drop=True)
    titles_clean = [strip_accents(str(t).lower()) for t in sub_df["thesis"]]
    years_list   = sub_df["year"].tolist()

    year_word, year_total = {}, {}
    for year, doc in zip(
        years_list,
        nlp.pipe(titles_clean, batch_size=512, n_process=1)
    ):
        toks = tokenize_doc(doc)
        year_word.setdefault(year, Counter()).update(toks)
        year_total[year] = year_total.get(year, 0) + len(toks)

    with open(out_path, "wb") as f:
        pickle.dump({"year_word": year_word, "year_total": year_total}, f)

print("All subject counts saved.")

Processing 63 subjects...


subjects: 100%|██████████| 63/63 [00:00<?, ?it/s]

All subject counts saved.


In [6]:
# ── 3. FLAT MODELLING TABLE ────────────────────────────────────────────────
# Columns: word, subject, year, count, total_title_words, share, specificity
# specificity = subject_share / global_share  (TF-IDF-style subject focus)

MODEL_PATH = OUT_DIR / "word_year_subject.parquet"

if MODEL_PATH.exists():
    print("word_year_subject.parquet already exists, skipping.")
else:
    records = []

    for subj in tqdm(all_subjects, desc="building flat table"):
        pkl_path = SUBJ_DIR / f"{subj}.pkl"
        if not pkl_path.exists():
            continue

        with open(pkl_path, "rb") as f:
            counts = pickle.load(f)

        year_word  = counts["year_word"]
        year_total = counts["year_total"]
        subj_total_words = sum(year_total.values())

        for year, word_counts in year_word.items():
            total = year_total[year]
            for word, count in word_counts.items():
                subject_share  = count / max(subj_total_words, 1)
                global_share   = global_counts[word] / max(global_total, 1)
                specificity    = subject_share / max(global_share, 1e-9)
                records.append({
                    "word":              word,
                    "subject":           subj,
                    "year":              year,
                    "count":             count,
                    "total_title_words": total,
                    "share":             count / max(total, 1),
                    "specificity":       specificity,
                })

    df_model = pd.DataFrame(records)
    df_model["year"]    = df_model["year"].astype(int)
    df_model["subject"] = df_model["subject"].astype(str).str.zfill(2)
    df_model["count"]   = df_model["count"].astype(int)

    df_model.to_parquet(MODEL_PATH, index=False)
    print(f"Saved {MODEL_PATH} — {len(df_model):,} rows, {df_model.shape[1]} columns")

word_year_subject.parquet already exists, skipping.


In [7]:
# ── 4. SUBJECT METADATA ────────────────────────────────────────────────────
# subject code → name + title count. Useful for labelling plots.

META_PATH = OUT_DIR / "subject_meta.parquet"

subject_meta = (
    df_en.dropna(subset=["subject_name"])
    .groupby("code")
    .agg(
        subject_name=("subject_name", lambda x: x.mode()[0]),
        n_titles=("thesis", "count"),
    )
    .reset_index()
    .rename(columns={"code": "subject"})
)

subject_meta.to_parquet(META_PATH, index=False)
print(f"Saved {META_PATH}")
print(subject_meta.sort_values("n_titles", ascending=False).head(10).to_string(index=False))

Saved data\subject_meta.parquet
subject                                           subject_name  n_titles
     68                                       Computer science     34030
     62                                             Statistics     21023
     91 Game theory, economics, social and behavioral sciences     18876
     65                                     Numerical analysis      7573
     60            Probability theory and stochastic processes      7342
     90          Operations research, mathematical programming      6410
     35                         Partial differential equations      6338
     92                     Biology and other natural sciences      6210
     11                                          Number theory      5760
     76                                        Fluid mechanics      5699


In [8]:
# ── 5. SANITY CHECK ────────────────────────────────────────────────────────
df_check = pd.read_parquet(MODEL_PATH)

print("Schema:")
print(df_check.dtypes)
print(f"\nRows:     {len(df_check):,}")
print(f"Subjects: {df_check['subject'].nunique()}")
print(f"Words:    {df_check['word'].nunique():,}")
print(f"Years:    {df_check['year'].min()} – {df_check['year'].max()}")
print("\nSample (subject 68, learning):")
print(
    df_check[(df_check["subject"] == "68") & (df_check["word"] == "learning")]
    .sort_values("year").tail(10).to_string(index=False)
)

Schema:
word                  object
subject               object
year                   int32
count                  int32
total_title_words      int64
share                float64
specificity          float64
dtype: object

Rows:     667,176
Subjects: 63
Words:    41,384
Years:    1681 – 2026

Sample (subject 68, learning):
    word subject  year  count  total_title_words    share  specificity
learning      68  2016     17               5846 0.002908     0.069197
learning      68  2017     22               5195 0.004235     0.089548
learning      68  2018     36               5254 0.006852     0.146534
learning      68  2019     24               5243 0.004578     0.097689
learning      68  2020     25               4713 0.005304     0.101760
learning      68  2021     31               4064 0.007628     0.126182
learning      68  2022     30               4302 0.006974     0.122111
learning      68  2023     42               4095 0.010256     0.170956
learning      68  2024     32    

In [ ]:
#!pip install gensim

In [9]:
# %% [setup]
import re
import os
import pickle
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from gensim.models.phrases import Phrases, Phraser
from tqdm.auto import tqdm

DATA_PATH = Path.cwd().parent.parent / "data" / "processed" / "dataset_filled_subjects_new.csv"
OUT_DIR   = Path("data")
SUBJ_DIR  = OUT_DIR / "subject_counts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBJ_DIR.mkdir(parents=True, exist_ok=True)

print("Paths ready")

# %% [load]
df_en = pd.read_pickle("data/keyword_trends_english.pkl")
df_en["year"] = df_en["year"].astype(int)
df_en["code"] = df_en["subject_code"].astype(int).astype(str).str.zfill(2)

print(f"Loaded {len(df_en):,} English titles")
print(f"Subjects: {df_en['code'].nunique()}")

# %% [spacy + bigram setup]
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

KEEP_POS = {"NOUN", "ADJ", "VERB", "PROPN"}

def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def tokenize_doc(doc):
    """Unigrams only -- bigram model applied after."""
    return [
        token.lemma_ for token in doc
        if token.pos_ in KEEP_POS
        and not token.is_stop
        and len(token.lemma_) >= 4
    ]

# -- train bigram detector on all titles --
BIGRAM_PATH = OUT_DIR / "bigram_model.pkl"

if BIGRAM_PATH.exists():
    print("Loading bigram model...")
    with open(BIGRAM_PATH, "rb") as f:
        bigram = pickle.load(f)
else:
    print("Training bigram model on all titles...")
    all_titles_clean = [strip_accents(str(t).lower()) for t in df_en["thesis"]]

    # first pass: get unigram token lists
    all_token_lists = []
    for doc in tqdm(nlp.pipe(all_titles_clean, batch_size=512, n_process=1),
                    total=len(all_titles_clean), desc="tokenizing for bigram training"):
        all_token_lists.append(tokenize_doc(doc))

    # train phrases model
    # min_count: bigram must appear at least 30 times
    # threshold: higher = fewer bigrams detected (PMI-like score)
    phrases = Phrases(all_token_lists, min_count=30, threshold=15)
    bigram  = Phraser(phrases)

    with open(BIGRAM_PATH, "wb") as f:
        pickle.dump(bigram, f)
    print(f"Saved bigram model to {BIGRAM_PATH}")

# preview: what bigrams were learned?
bigram_vocab = [k.decode() if isinstance(k, bytes) else k
                for k in bigram.phrasegrams.keys()]
print(f"Bigrams learned: {len(bigram_vocab)}")
print("Sample:", bigram_vocab[:20])

# -- final tokenizer used everywhere downstream --
def tokenize_with_bigrams(doc):
    unigrams = tokenize_doc(doc)
    return list(bigram[unigrams])   # e.g. ["hilbert", "scheme"] -> ["hilbert_scheme"]

# %% [1. global counts]
GLOBAL_PATH = OUT_DIR / "global_counts.parquet"

if GLOBAL_PATH.exists():
    print("global_counts.parquet already exists, loading...")
    _g            = pd.read_parquet(GLOBAL_PATH)
    global_counts = Counter(dict(zip(_g["lemma"], _g["count"])))
    global_total  = int(_g["count"].sum())
else:
    print("Building global counts...")
    global_counts = Counter()
    global_total  = 0

    all_titles_clean = [strip_accents(str(t).lower()) for t in df_en["thesis"]]
    for doc in tqdm(nlp.pipe(all_titles_clean, batch_size=512, n_process=1),
                    total=len(all_titles_clean), desc="global tokenize"):
        toks = tokenize_with_bigrams(doc)
        global_counts.update(toks)
        global_total += len(toks)

    pd.DataFrame(
        global_counts.most_common(),
        columns=["lemma", "count"]
    ).to_parquet(GLOBAL_PATH, index=False)
    print(f"Saved {GLOBAL_PATH}")

print(f"Global vocabulary: {len(global_counts):,} lemmas | {global_total:,} total tokens")


Paths ready
Loaded 208,150 English titles
Subjects: 63
Loading bigram model...
Bigrams learned: 632
Sample: ['singular_integral', 'viscous_fluid', 'hilbert_space', 'hypergeometric_function', 'periodic_orbit', 'differential_equation', 'cross_section', 'periodic_solution', 'second_order', 'partial_differential', 'porous_medium', 'ordinary_differential', 'runge_kutta', 'boundary_condition', 'delay_differential', 'banach_space', 'finite_difference', 'automorphism_group', 'tensor_product', 'numerical_solution']
global_counts.parquet already exists, loading...
Global vocabulary: 41,384 lemmas | 1,209,297 total tokens


In [10]:

# %% [2. per-subject counts]
SUBJ_DIR.mkdir(parents=True, exist_ok=True)  # add this line
print(f"Processing {len(all_subjects)} subjects...")
print(f"Processing {len(all_subjects)} subjects...")

for subj in tqdm(all_subjects, desc="subjects"):
    out_path = SUBJ_DIR / f"{subj}.pkl"
    if out_path.exists():
        continue

    sub_df       = df_en[df_en["code"] == subj].reset_index(drop=True)
    titles_clean = [strip_accents(str(t).lower()) for t in sub_df["thesis"]]
    years_list   = sub_df["year"].tolist()

    year_word, year_total = {}, {}
    for year, doc in zip(
        years_list,
        nlp.pipe(titles_clean, batch_size=512, n_process=1)
    ):
        toks = tokenize_with_bigrams(doc)
        year_word.setdefault(year, Counter()).update(toks)
        year_total[year] = year_total.get(year, 0) + len(toks)

    with open(out_path, "wb") as f:
        pickle.dump({"year_word": year_word, "year_total": year_total}, f)

print("All subject counts saved.")



Processing 63 subjects...
Processing 63 subjects...


subjects: 100%|██████████| 63/63 [00:00<?, ?it/s]

All subject counts saved.


In [11]:
# %% [3. flat modelling table]
MODEL_PATH = OUT_DIR / "word_year_subject.parquet"

if MODEL_PATH.exists():
    print("word_year_subject.parquet already exists, skipping.")
else:
    records = []

    for subj in tqdm(all_subjects, desc="building flat table"):
        pkl_path = SUBJ_DIR / f"{subj}.pkl"
        if not pkl_path.exists():
            continue

        with open(pkl_path, "rb") as f:
            counts = pickle.load(f)

        year_word        = counts["year_word"]
        year_total       = counts["year_total"]
        subj_total_words = sum(year_total.values())

        for year, word_counts in year_word.items():
            total = year_total[year]
            for word, count in word_counts.items():
                subject_share = count / max(subj_total_words, 1)
                global_share  = global_counts[word] / max(global_total, 1)
                specificity   = subject_share / max(global_share, 1e-9)
                records.append({
                    "word":              word,
                    "subject":           subj,
                    "year":              year,
                    "count":             count,
                    "total_title_words": total,
                    "share":             count / max(total, 1),
                    "specificity":       specificity,
                })

    df_model = pd.DataFrame(records)
    df_model["year"]    = df_model["year"].astype(int)
    df_model["subject"] = df_model["subject"].astype(str).str.zfill(2)
    df_model["count"]   = df_model["count"].astype(int)

    df_model.to_parquet(MODEL_PATH, index=False)
    print(f"Saved {MODEL_PATH} — {len(df_model):,} rows, {df_model.shape[1]} columns")




word_year_subject.parquet already exists, skipping.


In [12]:
# %% [4. subject metadata]
META_PATH = OUT_DIR / "subject_meta.parquet"

subject_meta = (
    df_en.dropna(subset=["subject_name"])
    .groupby("code")
    .agg(
        subject_name=("subject_name", lambda x: x.mode()[0]),
        n_titles=("thesis", "count"),
    )
    .reset_index()
    .rename(columns={"code": "subject"})
)

subject_meta.to_parquet(META_PATH, index=False)
print(f"Saved {META_PATH}")


Saved data\subject_meta.parquet


In [13]:
# %% [5. sanity check]
df_check = pd.read_parquet(MODEL_PATH)

print("Schema:")
print(df_check.dtypes)
print(f"\nRows:     {len(df_check):,}")
print(f"Subjects: {df_check['subject'].nunique()}")
print(f"Words:    {df_check['word'].nunique():,}")
print(f"Years:    {df_check['year'].min()} – {df_check['year'].max()}")

# check bigrams made it through
bigram_rows = df_check[df_check["word"].str.contains("_")]
print(f"\nBigram tokens: {bigram_rows['word'].nunique():,} unique")
print("Sample bigrams:")
print(bigram_rows.groupby("word")["count"].sum().nlargest(20).to_string())

Schema:
word                  object
subject               object
year                   int32
count                  int32
total_title_words      int64
share                float64
specificity          float64
dtype: object

Rows:     667,176
Subjects: 63
Words:    41,384
Years:    1681 – 2026

Bigram tokens: 1,128 unique
Sample bigrams:
word
differential_equation    1557
partial_differential     1056
time_series              1020
large_scale               916
finite_element            898
dynamical_system          866
high_dimensional          862
boundary_value            741
high_order                725
banach_space              692
machine_learning          681
optimal_control           667
neural_network            601
real_time                 595
numerical_solution        532
monte_carlo               503
second_order              498
united_states             483
markov_chain              440
statistical_inference     413
